# Evaluating RAG Agents with RAGAS

**RAGAS** (Retrieval Augmented Generation Assessment) is a framework for scoring RAG pipelines on metrics that single **BLEU/ROUGE** scores miss:

| Metric | Question |
|--------|----------|
| **Faithfulness** | Is the answer supported by retrieved context? |
| **Answer relevancy** | Does the answer address the user question? |
| **Context precision** | Are retrieved chunks relevant? |
| **Context recall** | Did retrieval find what ground truth needs? |

```
  User question
       │
       ▼
  ┌────────────┐     contexts[]     ┌────────────┐
  │ Retriever  │ ─────────────────► │ Generator  │ → answer
  └────────────┘                    └────────────┘
       │                                   │
       └─────────── RAGAS metrics ─────────┘
```

This notebook builds a **ShopCo Support RAG agent**, implements **RAGAS-style scorers** in plain Python, runs an **eval dataset**, compares **good vs bad retrieval**, and includes an optional cell for the real **ragas** library.

In [ ]:
"""
RAGAS evaluation lab — ShopCo Support RAG agent.
"""

import json
import math
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
USE_RAGAS_SDK = False  # set True if ragas package is installed


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_CORPUS: list[dict[str, str]] = [
    {"id": "pol-returns-01", "title": "Returns", "text": "Customers may return items within 30 days of delivery with receipt. Final-sale items cannot be returned. [pol-returns-01]"},
    {"id": "pol-shipping-02", "title": "Shipping", "text": "Free standard shipping on orders over $50. Express available at checkout. [pol-shipping-02]"},
    {"id": "pol-billing-03", "title": "Billing", "text": "Charges appear as SHOPCO* on statements. Partial refunds may take two billing cycles. [pol-billing-03]"},
    {"id": "faq-hours-05", "title": "Support Hours", "text": "Support hours Mon–Fri 9am–6pm ET. [faq-hours-05]"},
]

print(f"Corpus: {len(POLICY_CORPUS)} policy docs")

---

## 1. RAG Eval Requires Four Fields

Each eval row needs:

1. **`question`** — user query
2. **`contexts`** — list of retrieved chunk strings
3. **`answer`** — agent-generated response
4. **`ground_truth`** (optional) — reference answer for recall/correctness

In [ ]:
@dataclass
class RagEvalRow:
    row_id: str
    question: str
    contexts: list[str]
    answer: str
    ground_truth: str = ""

    def to_dict(self) -> dict[str, Any]:
        return {
            "question": self.question,
            "contexts": self.contexts,
            "answer": self.answer,
            "ground_truth": self.ground_truth,
        }


sample_row = RagEvalRow(
    "r1", "What is the return window?",
    [POLICY_CORPUS[0]["text"]],
    "You may return within 30 days. [pol-returns-01]",
    "Returns within 30 days with receipt.",
)
print(pretty(sample_row.to_dict()))

---

## 2. ShopCo Retriever — Keyword Scoring

In [ ]:
def tokenize(text: str) -> list[str]:
    return [t for t in re.split(r"\W+", text.lower()) if len(t) > 2]


def retrieve(query: str, corpus: list[dict[str, str]], top_k: int = 2) -> list[str]:
    terms = tokenize(query)
    scored: list[tuple[float, str]] = []
    for doc in corpus:
        hay = f"{doc['title']} {doc['text']}".lower()
        score = sum(1 for t in terms if t in hay) if terms else 0
        if score > 0:
            scored.append((score, doc["text"]))
    scored.sort(key=lambda x: -x[0])
    return [text for _, text in scored[:top_k]] if scored else [corpus[0]["text"]]


print(retrieve("return policy window", POLICY_CORPUS))

---

## 3. ShopCo RAG Agent — Retrieve + Compose Answer

In [ ]:
@dataclass
class RagAgentResult:
    question: str
    contexts: list[str]
    answer: str
    cited_ids: list[str]


def extract_citations(text: str) -> list[str]:
    return re.findall(r"\[([a-z]+-[a-z]+-\d+)\]", text)


def compose_answer(question: str, contexts: list[str]) -> str:
    """Rule-based generator — cites policy ids from context."""
    combined = " ".join(contexts)
    cites = extract_citations(combined)
    cite_str = f" [{cites[0]}]" if cites else ""
    if "return" in question.lower():
        m = re.search(r"(\d+)\s*days", combined)
        days = m.group(1) if m else "30"
        return f"You may return items within {days} days of delivery.{cite_str}"
    if "ship" in question.lower():
        return f"Free shipping applies on qualifying orders.{cite_str}"
    if "hour" in question.lower():
        return f"Support hours are Mon–Fri 9am–6pm ET.{cite_str}"
    return f"Based on our policies: {combined[:120]}...{cite_str}"


def shopco_rag_agent(question: str, retriever: Callable[..., list[str]] = retrieve) -> RagAgentResult:
    contexts = retriever(question, POLICY_CORPUS)
    answer = compose_answer(question, contexts)
    return RagAgentResult(question, contexts, answer, extract_citations(answer))


result = shopco_rag_agent("What is the return window?")
print(result.answer)

---

## 4. Faithfulness — Claims Supported by Context?

In [ ]:
def split_claims(answer: str) -> list[str]:
    parts = re.split(r"[.!?]\s+", answer)
    return [p.strip() for p in parts if len(p.strip()) > 5]


def claim_supported(claim: str, context: str) -> bool:
    claim_terms = set(tokenize(claim))
    ctx_terms = set(tokenize(context))
    if not claim_terms:
        return True
    overlap = claim_terms & ctx_terms
    return len(overlap) / len(claim_terms) >= 0.4


def score_faithfulness(answer: str, contexts: list[str]) -> float:
    ctx_blob = " ".join(contexts)
    claims = split_claims(answer)
    if not claims:
        return 1.0
    supported = sum(1 for c in claims if claim_supported(c, ctx_blob))
    return round(supported / len(claims), 2)


good = score_faithfulness(result.answer, result.contexts)
bad = score_faithfulness("Returns take 90 days and require a notarized letter.", result.contexts)
print(f"Faithful: {good} | Hallucinated: {bad}")

---

## 5. Answer Relevancy — Does Answer Match Question?

In [ ]:
def term_overlap(a: str, b: str) -> float:
    ta, tb = set(tokenize(a)), set(tokenize(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def score_answer_relevancy(question: str, answer: str) -> float:
    q_terms = set(tokenize(question))
    a_terms = set(tokenize(answer))
    if not q_terms:
        return 0.0
    direct = len(q_terms & a_terms) / len(q_terms)
    semantic_boost = 0.0
    if "return" in question.lower() and "return" in answer.lower():
        semantic_boost = 0.3
    if "ship" in question.lower() and "ship" in answer.lower():
        semantic_boost = 0.3
    if "hour" in question.lower() and ("hour" in answer.lower() or "9am" in answer.lower()):
        semantic_boost = 0.3
    return round(min(1.0, direct + semantic_boost), 2)


print("Relevant:", score_answer_relevancy(result.question, result.answer))
print("Irrelevant:", score_answer_relevancy(result.question, "Billing charges appear as SHOPCO*."))

---

## 6. Context Precision — Retrieved Chunks On-Topic?

In [ ]:
def chunk_relevant_to_question(question: str, chunk: str) -> bool:
    q = question.lower()
    c = chunk.lower()
    if "return" in q:
        return "return" in c or "pol-returns" in c
    if "ship" in q:
        return "ship" in c or "pol-shipping" in c
    if "hour" in q or "support" in q:
        return "hour" in c or "faq-hours" in c
    return term_overlap(question, chunk) > 0.1


def score_context_precision(question: str, contexts: list[str]) -> float:
    if not contexts:
        return 0.0
    relevant = sum(1 for c in contexts if chunk_relevant_to_question(question, c))
    return round(relevant / len(contexts), 2)


good_ctx = retrieve("return window", POLICY_CORPUS)
bad_ctx = [POLICY_CORPUS[2]["text"], POLICY_CORPUS[3]["text"]]  # billing + hours for return Q
print("Good precision:", score_context_precision("return window", good_ctx))
print("Bad precision:", score_context_precision("return window", bad_ctx))

---

## 7. Context Recall — Context Covers Ground Truth?

In [ ]:
def score_context_recall(contexts: list[str], ground_truth: str) -> float:
    if not ground_truth:
        return 1.0
    gt_terms = set(tokenize(ground_truth))
    if not gt_terms:
        return 1.0
    ctx_terms = set(tokenize(" ".join(contexts)))
    overlap = len(gt_terms & ctx_terms) / len(gt_terms)
    return round(overlap, 2)


gt = "Returns within 30 days with receipt."
print("Good recall:", score_context_recall(good_ctx, gt))
print("Bad recall:", score_context_recall(bad_ctx, gt))

---

## 8. RAGAS Report — Aggregate Scorer

In [ ]:
@dataclass
class RagasScores:
    faithfulness: float
    answer_relevancy: float
    context_precision: float
    context_recall: float

    def average(self) -> float:
        return round((self.faithfulness + self.answer_relevancy + self.context_precision + self.context_recall) / 4, 2)

    def passes(self, threshold: float = 0.75) -> bool:
        return self.average() >= threshold and self.faithfulness >= 0.7


def evaluate_row(row: RagEvalRow) -> RagasScores:
    return RagasScores(
        faithfulness=score_faithfulness(row.answer, row.contexts),
        answer_relevancy=score_answer_relevancy(row.question, row.answer),
        context_precision=score_context_precision(row.question, row.contexts),
        context_recall=score_context_recall(row.contexts, row.ground_truth),
    )


print(pretty(evaluate_row(sample_row)))

---

## 9. Golden Eval Dataset — ShopCo

In [ ]:
GOLDEN_DATASET: list[dict[str, str]] = [
    {"question": "What is the return window?", "ground_truth": "Returns within 30 days with receipt."},
    {"question": "When is free shipping available?", "ground_truth": "Free shipping on orders over $50."},
    {"question": "What are support hours?", "ground_truth": "Mon–Fri 9am–6pm ET."},
]


def build_eval_rows(agent_fn: Callable[[str], RagAgentResult]) -> list[RagEvalRow]:
    rows: list[RagEvalRow] = []
    for i, item in enumerate(GOLDEN_DATASET):
        r = agent_fn(item["question"])
        rows.append(RagEvalRow(
            f"eval-{i+1}", item["question"], r.contexts, r.answer, item["ground_truth"],
        ))
    return rows


eval_rows = build_eval_rows(shopco_rag_agent)
print(f"Built {len(eval_rows)} eval rows")

---

## 10. Run Full RAGAS Suite

In [ ]:
@dataclass
class SuiteResult:
    row_id: str
    question: str
    scores: RagasScores
    passed: bool


def run_ragas_suite(rows: list[RagEvalRow], threshold: float = 0.75) -> dict[str, Any]:
    results: list[SuiteResult] = []
    for row in rows:
        scores = evaluate_row(row)
        results.append(SuiteResult(row.row_id, row.question, scores, scores.passes(threshold)))
    passed = sum(1 for r in results if r.passed)
    avg = RagasScores(
        faithfulness=round(sum(r.scores.faithfulness for r in results) / len(results), 2),
        answer_relevancy=round(sum(r.scores.answer_relevancy for r in results) / len(results), 2),
        context_precision=round(sum(r.scores.context_precision for r in results) / len(results), 2),
        context_recall=round(sum(r.scores.context_recall for r in results) / len(results), 2),
    )
    return {
        "total": len(results),
        "passed": passed,
        "pass_rate": round(passed / len(results), 2),
        "averages": avg,
        "rows": [{"id": r.row_id, "q": r.question[:40], "passed": r.passed, "avg": r.scores.average()} for r in results],
    }


print(pretty(run_ragas_suite(eval_rows)))

---

## 11. Bad Retriever Ablation — Diagnose Retrieval

In [ ]:
def bad_retriever(query: str, corpus: list[dict[str, str]], top_k: int = 2) -> list[str]:
    """Always returns billing doc — simulates broken embedding index."""
    return [corpus[2]["text"]]


def shopco_rag_bad(question: str) -> RagAgentResult:
    return shopco_rag_agent(question, retriever=bad_retriever)


good_suite = run_ragas_suite(eval_rows)
bad_rows = build_eval_rows(shopco_rag_bad)
bad_suite = run_ragas_suite(bad_rows)

print("Good retriever pass_rate:", good_suite["pass_rate"])
print("Bad retriever pass_rate:", bad_suite["pass_rate"])
print("Good context_precision:", good_suite["averages"].context_precision)
print("Bad context_precision:", bad_suite["averages"].context_precision)

---

## 12. Hallucination Case — High Relevancy, Low Faithfulness

In [ ]:
halluc_row = RagEvalRow(
    "halluc-1",
    "What is the return window?",
    [POLICY_CORPUS[0]["text"]],
    "You can return within 90 days, no receipt needed. [pol-returns-01]",
    "Returns within 30 days with receipt.",
)
halluc_scores = evaluate_row(halluc_row)
print(pretty(halluc_scores))
print("Relevancy ok but faithfulness fails:", halluc_scores.answer_relevancy > 0.5 and halluc_scores.faithfulness < 0.7)

---

## 13. Agent + RAG — Trace Contexts in Observability

Attach RAGAS inputs to traces for debugging:

In [ ]:
@dataclass
class RagTraceRecord:
    trace_id: str
    question: str
    contexts: list[str]
    answer: str
    ragas: RagasScores
    ts: str = field(default_factory=utc_now)


def rag_run_with_eval(question: str) -> RagTraceRecord:
    r = shopco_rag_agent(question)
    row = RagEvalRow("live", r.question, r.contexts, r.answer)
    scores = evaluate_row(row)
    return RagTraceRecord(f"tr-{uuid.uuid4().hex[:8]}", r.question, r.contexts, r.answer, scores)


rec = rag_run_with_eval("return policy?")
print(rec.trace_id, "faithfulness:", rec.ragas.faithfulness)

---

## 14. CI Gate — Block Deploy on RAGAS Regression

In [ ]:
RAGAS_GATES = {
    "faithfulness_min": 0.70,
    "context_precision_min": 0.65,
    "pass_rate_min": 0.80,
}


def ci_gate(suite: dict[str, Any]) -> dict[str, Any]:
    avg = suite["averages"]
    checks = {
        "faithfulness": avg.faithfulness >= RAGAS_GATES["faithfulness_min"],
        "context_precision": avg.context_precision >= RAGAS_GATES["context_precision_min"],
        "pass_rate": suite["pass_rate"] >= RAGAS_GATES["pass_rate_min"],
    }
    return {"deploy_ok": all(checks.values()), "checks": checks}


print("Good retriever:", ci_gate(good_suite))
print("Bad retriever:", ci_gate(bad_suite))

---

## 15. Optional — Real RAGAS SDK

Set `USE_RAGAS_SDK = True` after installing `ragas` and `datasets`.

In [ ]:
def try_ragas_sdk() -> dict[str, Any]:
    try:
        from datasets import Dataset
    except ImportError:
        return {"status": "datasets_not_installed"}
    try:
        from ragas import evaluate
        from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    except ImportError:
        return {"status": "ragas_not_installed", "hint": "pip install ragas datasets"}

    data = Dataset.from_list([
        {"question": r.question, "contexts": r.contexts, "answer": r.answer, "ground_truth": r.ground_truth}
        for r in eval_rows[:2]
    ])
    return {
        "status": "ready",
        "note": "Call evaluate(data, metrics=[faithfulness, ...]) with LLM API key configured",
        "rows": len(data),
    }


if USE_RAGAS_SDK:
    print(pretty(try_ragas_sdk()))
else:
    print("Simulation mode — plain Python RAGAS scorers above.")

---

## 16. Optional Live LLM Generator

Set `USE_LIVE_LLM = True` to generate answers with `gpt-4o-mini`, then score with local RAGAS.

In [ ]:
def generate_answer_live(question: str, contexts: list[str]) -> str:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    ctx = "\n".join(contexts)
    resp = llm.invoke([
        SystemMessage(content="Answer only from context. Cite [pol-*] ids."),
        HumanMessage(content=f"Context:\n{ctx}\n\nQuestion: {question}"),
    ])
    return str(resp.content)


q = "What is the return window?"
ctx = retrieve(q, POLICY_CORPUS)
if USE_LIVE_LLM:
    ans = generate_answer_live(q, ctx)
    print("Scores:", evaluate_row(RagEvalRow("live", q, ctx, ans, "Returns within 30 days.")))
else:
    print("Offline:", compose_answer(q, ctx))

---

## 17. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| ROUGE-only eval | High score, wrong facts | Add **faithfulness** |
| No ground truth | Can't measure recall | Golden dataset |
| Eval without contexts | Can't debug retrieval | Log contexts in trace |
| One metric only | Pass relevancy, fail faith | Full RAGAS suite |
| Skip ablation | Can't isolate retrieval bug | Bad retriever test |

---

## 18. Quiz

1. What four fields does a RAG eval row need?
2. What does faithfulness measure?
3. How does context precision differ from context recall?
4. Why run a bad-retriever ablation?
5. Which metric catches "90 days" when policy says 30?

---

## 19. Summary

**RAGAS** evaluates retrieval and generation together: faithfulness, answer relevancy, context precision, and context recall. ShopCo demos showed golden dataset runs, bad-retriever ablation, hallucination detection, and CI gates.

Log **contexts** in traces, run RAGAS in CI, and block deploy when faithfulness or precision regresses. Use the real **ragas** library in production for LLM-judge variants of these metrics.